# Tutorial 9: Train NicheTrans on embryonic mouse brain data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_hd import *
from datasets.data_manager_MISAR_seq import ATAC_RNA_Seq

from utils.utils import *
from utils.utils_training_embryonic_mouse_brain import *
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_MISAR_seq.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.2, dropout_rate=0.1, n_source=3000, workers=4, knn_smooth=True, peak_threshold=0.05, hvg_gene=1500, adata_path='/mnt/datadisk0/Processed_DATA/2023_nm_MISAR_seq', max_epoch=20, stepsize=10, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = ATAC_RNA_Seq(peak_threshold=args.peak_threshold, hvg_gene=args.hvg_gene, adata_path=args.adata_path, RNA2ATAC=True, knn_smoothing=args.knn_smooth)
trainloader, testloader = embryonic_mouse_brain(args, dataset)

# create the model
source_dimension, target_dimension = len(dataset.source_panel), len(dataset.target_panel)
model_kwargs = dict(
    source_length=source_dimension,
    target_length=target_dimension,
    noise_rate=args.noise_rate,
    dropout_rate=args.dropout_rate,
    n_spot_types=getattr(dataset, 'n_spot_types', 1),
)
model = NicheTrans(**model_kwargs)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)


  Slice e13: Leiden found 6 local clusters (sizes: [472, 408, 402, 300, 117, 78])
  Slice e15: Leiden found 10 local clusters (sizes: [430, 307, 276, 252, 234, 163, 111, 99, 45, 32])
  Slice e18: Leiden found 7 local clusters (sizes: [652, 456, 453, 202, 149, 121, 96])
  Slice e15: matched 6/10 local clusters into the global space
  Slice e18: matched 7/7 local clusters into the global space
  Cross-slice alignment complete: 10 global cell types
------Calculating spatial graph...
The graph contains 17032 edges, 2129 cells.
8.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 14216 edges, 1777 cells.
8.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 15592 edges, 1949 cells.
8.0000 neighbors per cell on average.
source size 1500
target size 34323
=> Spatial atac-rna Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |   3906 spo

### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.BCELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train_binary(model, criterion, optimizer, trainloader, device=device)
    if args.stepsize > 0: scheduler.step()
    ################

# test_binary(args, model, testloader)
test_regression(model, testloader, if_sigmoid=True, device=device)
torch.save(model.state_dict(), 'NicheTrans_embryonic_mouse_brain_rna2atac.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20
Batch 122/122	 Loss 0.633284 (0.669029)
==> Epoch 2/20
Batch 122/122	 Loss 0.575199 (0.599787)
==> Epoch 3/20
Batch 122/122	 Loss 0.538650 (0.561154)
==> Epoch 4/20
Batch 122/122	 Loss 0.521759 (0.537328)
==> Epoch 5/20
Batch 122/122	 Loss 0.523405 (0.520048)
==> Epoch 6/20
Batch 122/122	 Loss 0.504228 (0.510955)
==> Epoch 7/20
Batch 122/122	 Loss 0.478920 (0.499382)
==> Epoch 8/20
Batch 122/122	 Loss 0.479052 (0.490031)
==> Epoch 9/20
Batch 122/122	 Loss 0.480614 (0.479816)
==> Epoch 10/20
Batch 122/122	 Loss 0.474110 (0.474557)
==> Epoch 11/20
Batch 122/122	 Loss 0.474296 (0.473638)
==> Epoch 12/20
Batch 122/122	 Loss 0.469452 (0.471247)
==> Epoch 13/20
Batch 122/122	 Loss 0.477408 (0.470463)
==> Epoch 14/20
Batch 122/122	 Loss 0.478184 (0.468779)
==> Epoch 15/20
Batch 122/122	 Loss 0.468254 (0.468113)
==> Epoch 16/20
Batch 122/122	 Loss 0.483737 (0.467924)
==> Epoch 17/20
Batch 122/122	 Loss 0.478868 (0.467449)
==> Epoch 18/20
Batch 122/122	 Loss 0.459328 (0.466579)
=